# GNNHAR-IV Full Colab Run

This notebook runs the full GNNHAR-IV empirical analysis on Colab. Code and input data are cloned from GitHub, so no Google Drive mount is required. Outputs are written locally under `/content/gnnhar_outputs` and then uploaded to Google Drive with the Drive API using a service account.

Before running, add these Colab Secrets:

- `GITHUB_TOKEN`: a GitHub token with read access to the private `easygl1der/GNNHAR` repository.
- `GDRIVE_SERVICE_ACCOUNT_JSON`: the full service-account JSON key. Share the destination Drive folder with the service-account email.
- `GDRIVE_OUTPUT_FOLDER_ID`: the destination Google Drive folder ID.

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys

from google.colab import userdata

REPO_OWNER = 'easygl1der'
REPO_NAME = 'GNNHAR'
BRANCH = '2026-06-01'
REPO_DIR = pathlib.Path('/content/GNNHAR')
OUTPUT_DIR = pathlib.Path('/content/gnnhar_outputs')

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Missing Colab Secret: GITHUB_TOKEN')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

repo_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH, repo_url, str(REPO_DIR),
], check=True)

print('cloned:', REPO_DIR)
print('output_dir:', OUTPUT_DIR)

In [ ]:
import subprocess
import sys

mods = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'torch', 'scipy', 'googleapiclient', 'google.oauth2']
packages = {
    'sklearn': 'scikit-learn',
    'googleapiclient': 'google-api-python-client',
    'google.oauth2': 'google-auth',
}
missing = []
for mod in mods:
    try:
        __import__(mod)
    except Exception:
        missing.append(mod)
print('missing:', missing)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *[packages.get(m, m) for m in missing]])

In [ ]:
subprocess.run([
    'python', str(REPO_DIR / 'scripts/analysis/gnnhar_iv_pipeline.py'),
    '--data-dir', str(REPO_DIR / 'experiments/dow30/data'),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '250',
], check=True)

In [ ]:
import json
import pathlib

for path in sorted(OUTPUT_DIR.glob('**/*')):
    if path.is_file():
        print(path)

print('\nmetadata:')
print(json.dumps(json.loads((OUTPUT_DIR / 'run_metadata.json').read_text()), indent=2)[:2000])

In [ ]:
import pandas as pd

pd.read_csv(OUTPUT_DIR / 'tables/model_losses.csv').head(20)

In [ ]:
import json
import mimetypes

from google.colab import userdata
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

service_account_json = userdata.get('GDRIVE_SERVICE_ACCOUNT_JSON')
folder_id = userdata.get('GDRIVE_OUTPUT_FOLDER_ID')
if not service_account_json:
    raise RuntimeError('Missing Colab Secret: GDRIVE_SERVICE_ACCOUNT_JSON')
if not folder_id:
    raise RuntimeError('Missing Colab Secret: GDRIVE_OUTPUT_FOLDER_ID')

info = json.loads(service_account_json)
creds = service_account.Credentials.from_service_account_info(
    info, scopes=['https://www.googleapis.com/auth/drive']
)
drive = build('drive', 'v3', credentials=creds)

def ensure_folder(name, parent_id):
    escaped = name.replace("'", "\\'")
    query = (
        "mimeType='application/vnd.google-apps.folder' "
        f"and name='{escaped}' and '{parent_id}' in parents and trashed=false"
    )
    res = drive.files().list(q=query, fields='files(id, name)', pageSize=1).execute()
    files = res.get('files', [])
    if files:
        return files[0]['id']
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
    return drive.files().create(body=meta, fields='id').execute()['id']

def upload_file(path, parent_id):
    mime_type = mimetypes.guess_type(path.name)[0] or 'application/octet-stream'
    meta = {'name': path.name, 'parents': [parent_id]}
    media = MediaFileUpload(str(path), mimetype=mime_type, resumable=True)
    return drive.files().create(body=meta, media_body=media, fields='id, name').execute()

def upload_tree(root, parent_id):
    folder_cache = {root: parent_id}
    for path in sorted(root.rglob('*')):
        parent = folder_cache[path.parent]
        if path.is_dir():
            folder_cache[path] = ensure_folder(path.name, parent)
        elif path.is_file():
            uploaded = upload_file(path, parent)
            print('uploaded:', path.relative_to(root), uploaded['id'])

run_folder = ensure_folder('gnnhar_colab_rerun_outputs', folder_id)
upload_tree(OUTPUT_DIR, run_folder)
print('Drive upload folder id:', run_folder)